In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ai-tudy/pipeline_detected_family_image

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ai-tudy/pipeline_detected_family_image


Чеклист требований к проекту:
1. Постановка задачи
Что решаем, кто пользователь, зачем это нужно

2. Данные
Откуда взяты данные, сколько объектов, какие признаки/классы/документы

3. Baseline
Простое решение, с которым сравниваемся

4. Основной подход
Модель, агент, приложение или экспериментальная система

5. Метрики
Подходящие для задачи:
accuracy, F1, ROC-AUC, MSE, MAE, retrieval accuracy
hallucination rate, JSON accuracy, human eval и т.д

6. Эксперименты
Хотя бы 2–3 осмысленных сравнения

7. Анализ ошибок
Примеры, где система ошибается, и почему

8. Выводы
Что получилось, что не получилось, что можно улучшить

9. Код и воспроизводимость
README, requirements (а лучше Docker), инструкция запуска

## 1. Постановка проблемы.

Задача: автоматизировать отбор семейных фотографий для экономии места на устройстве: из неструктурированного архива выделить подмножество «корректных» фото (без дефектов качества и неудачных кадров с людьми), чтобы пользователь мог перенести на новый диск только их.

Целевая аудитория: непрофильные пользователи (обычные люди), которые много фотографируют близких; не имеют технических навыков, не хотят вручную сортировать тысячи фото.

Контекст использования:

 - Пользователь загружает папку с фото
 - Система обрабатывает пакетно (batch) и формирует две папки: good/ и defective/.
 - Для спорных случаев (неоднозначные кадры) система может помечать их тегом review/ — на усмотрение пользователя.

## 2. Данные

Данные я буду брать из базы семейных и личных фотографий, а также из дополнитьльных из интернета.

Дполнительные:
 - COCO 2017 (https://www.kaggle.com/datasets/awsaf49/coco-2017-dataset);
 - Blur dataset (https://www.kaggle.com/datasets/kwentar/blur-dataset)

Всего будет несколько признаков: размытые и не размытые для первой модели, "есть люди" и "нет людей" для второй. Остальные признаки: закрытие глаз, увод взгляда, открытый рот, искривление губ - для третьей модели

Получение датасета с людьми

Из kaggle в датасете COCO 2017 скачиваем только instances_train2017.json

In [ ]:
import sys
sys.path.insert(0, './utils/')
from downloading_from_coco_2017 import download_image_from_coco_2017_for_model_has_human
from dataset import ConvDataset
from trainer import train_model, get_y_true_pred
from other_utils import get_model_resnet18, view_classification_report, load_model, save_json
from PIL import Image
from pathlib import Path
import pandas as pd
import json
import torch
import torch.nn as nn
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import os
from tqdm import tqdm
from copy import deepcopy
from sklearn.metrics import f1_score, classification_report



Скачивание изображений с COCO 2017

In [ ]:
len_train_sample = 1000
len_test_sample = int(len_train_sample * 0.3)
download_image_from_coco_2017_for_model_has_human(len_train_sample, len_test_sample)


Основная работа по обучении модели.

In [ ]:
root_dir_train = 'data/train/coco2017'
root_dir_val = 'data/val/coco2017'

dataset_train = ConvDataset(root_dir_train)
dataset_val = ConvDataset(root_dir_val)


In [ ]:
train_loader = DataLoader(dataset_train, batch_size=16, shuffle=True, num_workers=2, pin_memory=True, prefetch_factor=2)
val_loader = DataLoader(dataset_val, batch_size=16, shuffle=False, num_workers=2, pin_memory=True, prefetch_factor=2)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_model_resnet18(num_classes=2).to(device)

criterion = nn.CrossEntropyLoss()  # требует target: torch.long
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=4,
    eta_min=1e-7
)

In [ ]:
path_save = 'models/has_human/best_model_has_human.pth'
history = train_model(model, train_loader, val_loader, path_save, criterion, optimizer, scheduler, device=device)



Epoch 1/15


Test: 100%|██████████| 36/36 [00:05<00:00,  6.38it/s]


Train Loss: 0.7095 | Train Acc: 0.6535
Val   Loss: 0.5796 | Val   Acc: 0.6842
--------------------
--------------------

Epoch 2/15


Test: 100%|██████████| 36/36 [00:04<00:00,  8.21it/s]


++++++++++++++++++++++++++++
Early Stopping: 1 / 5 эпох без улучшений. (Лучший лосс: 0.57956231895246, Текущий лосс: 0.641758788468545)
++++++++++++++++++++++++++++
Train Loss: 0.5246 | Train Acc: 0.7520
Val   Loss: 0.6418 | Val   Acc: 0.6667
--------------------
--------------------

Epoch 3/15


...

++++++++++++++++++++++++++++
Early Stopping: 3 / 5 эпох без улучшений. (Лучший лосс: 0.4146933459399039, Текущий лосс: 0.5412213413338912)
++++++++++++++++++++++++++++
Train Loss: 0.3471 | Train Acc: 0.8545
Val   Loss: 0.5412 | Val   Acc: 0.7754
--------------------
--------------------

Epoch 8/15


Test: 100%|██████████| 36/36 [00:04<00:00,  7.76it/s]


++++++++++++++++++++++++++++
Early Stopping: 4 / 5 эпох без улучшений. (Лучший лосс: 0.4146933459399039, Текущий лосс: 0.6887616243278771)
++++++++++++++++++++++++++++
Train Loss: 0.4113 | Train Acc: 0.8272
Val   Loss: 0.6888 | Val   Acc: 0.6667
--------------------
--------------------

Epoch 9/15


Test: 100%|██████████| 36/36 [00:04<00:00,  8.59it/s]

++++++++++++++++++++++++++++
Early Stopping: 5 / 5 эпох без улучшений. (Лучший лосс: 0.4146933459399039, Текущий лосс: 1.2530239142869648)
++++++++++++++++++++++++++++
Train Loss: 0.4604 | Train Acc: 0.7871
Val   Loss: 1.2530 | Val   Acc: 0.5842
--------------------
Сработала рання остановка!


Тестирование на своих данных

In [ ]:
root_dir_test = 'data/test/coco2017'
dataset_test = ConvDataset(root_dir_test)
test_loader = DataLoader(dataset_test, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
model = load_model("models/has_human/best_model_has_human.pth")

In [ ]:
y_true, y_pred = get_y_true_pred(model, test_loader, str(device))

Test: 100%|██████████| 13/13 [00:22<00:00,  1.76s/it]


In [ ]:
view_classification_report(y_true, y_pred, target_names=dataset_test.class_name_list)

               precision    recall  f1-score   support

    has_human       0.90      0.85      0.88        99
has_not_human       0.86      0.91      0.88       100

     accuracy                           0.88       199
    macro avg       0.88      0.88      0.88       199
 weighted avg       0.88      0.88      0.88       199



In [ ]:
save_json(dataset_test.dict_class_label, 'labels/label_has_human.json')

JSON успешно сохранен по пути: labels/label_has_human.json
